# 03 | Ten-classifier benchmark and evaluation
**Purpose.** Reproduce the manuscript's three original feature configurations and ten models on the **same** balanced 80/20 split (28,000 train / 7,000 test for the original data). Five-fold stratified cross-validation is performed only on training sources. Median imputation and any scaling are fitted within each training fold.

**Default:** `run_full_benchmark=true` includes nine scikit-learn-compatible classifiers and the TensorFlow DNN. All hyperparameters are read from `config/experiment.json`. To test the pipeline quickly, turn off `run_full_benchmark`, `run_dnn`, and `run_cross_validation` (not publication-equivalent).

**Outputs:** model files, predictions, class reports, and metric tables. Training records and the hold-out membership are saved for later notebooks.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd().resolve()
if not (ROOT / "config" / "experiment.json").exists():
    ROOT = ROOT.parent
if not (ROOT / "config" / "experiment.json").exists():
    raise FileNotFoundError("Start Jupyter from the repository root or notebooks/ folder.")
sys.path.insert(0, str(ROOT))
import numpy as np
import pandas as pd
from IPython.display import display
from src.gaia_pipeline import *
cfg = load_config(ROOT)
np.random.seed(cfg["random_state"])
print("Repository:", ROOT)
import joblib
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report
from sklearn.base import clone
from src.gaia_pipeline import make_base_pipeline, make_model, build_dnn, metric_dict
balanced = read_object(ROOT, "preprocessed.joblib")["balanced"]
model_dir = output_dir(ROOT) / "models"
model_dir.mkdir(exist_ok=True)
y_encoder = LabelEncoder().fit(balanced["SpType-ELS"])
y = y_encoder.transform(balanced["SpType-ELS"])
train_idx, test_idx = split_indices(balanced, cfg)
y_train, y_test = y[train_idx], y[test_idx]
assert set(balanced.iloc[train_idx]["Source"]).isdisjoint(set(balanced.iloc[test_idx]["Source"]))
print("Training:", len(train_idx), "| Held-out:", len(test_idx), "| Classes:", list(y_encoder.classes_))

## Original classifier parameters
All recorded settings are in `config/experiment.json`: LightGBM, XGBoost, Decision Tree, Gradient Boosting, SVM, Random Forest, KNN, MLP, CatBoost, and DNN. In the full benchmark, each classical model receives five CV folds, followed by an independent fit on all 28,000 training sources. Keras uses the training-fold procedure in the supplied source notebook.

In [ ]:
CLASSICAL_MODELS = ["LightGBM", "XGBoost", "DecisionTree", "GradientBoosting", "SVM", "RandomForest", "KNN", "MLP", "CatBoost"]
selected_models = CLASSICAL_MODELS if cfg["run_full_benchmark"] else list(BASE_MODELS)
print("Models:", selected_models + (["DNN"] if cfg["run_full_benchmark"] and cfg["run_dnn"] else []))
print("CV:", cfg["run_cross_validation"])
display(pd.DataFrame(cfg["hyperparameters"]).T)

## Train and evaluate classical models
Each representation uses exactly the same source indices. The `model.joblib` files are created by scikit-learn's Pipeline, including the fitted median imputer and any scaler; no test-set statistics enter preprocessing.

In [ ]:
benchmark = {"classes": list(y_encoder.classes_), "train_idx": train_idx, "test_idx": test_idx, "y_train": y_train, "y_test": y_test, "by_feature": {}}
rows = []
for feature_name, columns in FEATURE_SETS.items():
    X_train, X_test = balanced.iloc[train_idx][columns], balanced.iloc[test_idx][columns]
    predictions, probabilities = {}, {}
    (model_dir / feature_name).mkdir(exist_ok=True)
    for model_name in selected_models:
        print(f"{feature_name} | {model_name}")
        pipe = make_base_pipeline(model_name, cfg)
        if cfg["run_cross_validation"]:
            cv = StratifiedKFold(n_splits=cfg["cv_folds"], shuffle=True, random_state=cfg["random_state"])
            cv_scores = cross_val_score(pipe, X_train, y_train, cv=cv, scoring="f1_macro", n_jobs=1)
        else:
            cv_scores = np.array([])
        pipe.fit(X_train, y_train)
        prediction = np.asarray(pipe.predict(X_test)).ravel().astype(int)
        predictions[model_name] = prediction
        probabilities[model_name] = pipe.predict_proba(X_test)
        metrics = metric_dict(y_test, prediction)
        rows.append({"Feature set": feature_name, "Model": model_name, "cv_f1_macro_mean": np.mean(cv_scores) if len(cv_scores) else np.nan, "cv_f1_macro_std": np.std(cv_scores) if len(cv_scores) else np.nan, **metrics})
        joblib.dump(pipe, model_dir / feature_name / f"{model_name}.joblib")
        print(f"  macro-F1={metrics['f1_macro']:.4f} | MCC={metrics['mcc']:.4f}")
    benchmark["by_feature"][feature_name] = {"columns": columns, "predictions": predictions, "probabilities": probabilities}

## TensorFlow DNN
The three-layer DNN (256/128/64 units, batch normalization and dropout 0.30/0.30/0.20) follows the supplied research notebook. Each CV fold uses 40 maximum epochs; final training uses a stratified 10% validation partition and 100 maximum epochs, with early stopping on validation loss. The final DNN model and its fitted preprocessing objects are saved separately.

In [ ]:
if cfg["run_full_benchmark"] and cfg["run_dnn"]:
    import tensorflow as tf
    from tensorflow.keras.callbacks import EarlyStopping
    dnn_cfg = cfg["hyperparameters"]["DNN"]
    for feature_name, columns in FEATURE_SETS.items():
        print(f"DNN | {feature_name}")
        X_train, X_test = balanced.iloc[train_idx][columns], balanced.iloc[test_idx][columns]
        cv_scores = []
        if cfg["run_cross_validation"]:
            cv = StratifiedKFold(n_splits=cfg["cv_folds"], shuffle=True, random_state=cfg["random_state"])
            for tr, va in cv.split(X_train, y_train):
                imp, scaler = SimpleImputer(strategy="median"), StandardScaler()
                x_tr = scaler.fit_transform(imp.fit_transform(X_train.iloc[tr])).astype("float32")
                x_va = scaler.transform(imp.transform(X_train.iloc[va])).astype("float32")
                tf.keras.backend.clear_session(); tf.keras.utils.set_random_seed(cfg["random_state"])
                network = build_dnn(x_tr.shape[1], len(y_encoder.classes_), cfg)
                callback = EarlyStopping(monitor="val_loss", patience=dnn_cfg["early_stopping_patience"], restore_best_weights=True)
                network.fit(x_tr, y_train[tr].astype("int32"), validation_data=(x_va, y_train[va].astype("int32")), epochs=dnn_cfg["cv_epochs"], batch_size=dnn_cfg["batch_size"], callbacks=[callback], verbose=0)
                cv_scores.append(metric_dict(y_train[va], network.predict(x_va, verbose=0).argmax(axis=1))["f1_macro"])
        x_fit, x_val, y_fit, y_val = train_test_split(X_train, y_train, test_size=dnn_cfg["validation_fraction"], stratify=y_train, random_state=cfg["random_state"])
        imp, scaler = SimpleImputer(strategy="median"), StandardScaler()
        fit_array = scaler.fit_transform(imp.fit_transform(x_fit)).astype("float32")
        val_array = scaler.transform(imp.transform(x_val)).astype("float32")
        test_array = scaler.transform(imp.transform(X_test)).astype("float32")
        tf.keras.backend.clear_session(); tf.keras.utils.set_random_seed(cfg["random_state"])
        network = build_dnn(fit_array.shape[1], len(y_encoder.classes_), cfg)
        callback = EarlyStopping(monitor="val_loss", patience=dnn_cfg["early_stopping_patience"], restore_best_weights=True)
        network.fit(fit_array, y_fit.astype("int32"), validation_data=(val_array, y_val.astype("int32")), epochs=dnn_cfg["final_epochs"], batch_size=dnn_cfg["batch_size"], callbacks=[callback], verbose=0)
        probability = network.predict(test_array, verbose=0)
        prediction = probability.argmax(axis=1)
        benchmark["by_feature"][feature_name]["predictions"]["DNN"] = prediction
        benchmark["by_feature"][feature_name]["probabilities"]["DNN"] = probability
        rows.append({"Feature set": feature_name, "Model": "DNN", "cv_f1_macro_mean": np.mean(cv_scores) if cv_scores else np.nan, "cv_f1_macro_std": np.std(cv_scores) if cv_scores else np.nan, **metric_dict(y_test, prediction)})
        network.save(model_dir / feature_name / "DNN.keras")
        joblib.dump({"imputer": imp, "scaler": scaler}, model_dir / feature_name / "DNN_preprocessing.joblib")
else:
    print("DNN skipped by configuration. Enable run_full_benchmark and run_dnn for all ten models.")

## Persist the experiment record
This result file stores the *actual* held-out labels, source indices, predictions, probabilities and feature lists. Later notebooks do not retrain the benchmark unnecessarily. The original manuscript's complete benchmark requires all ten classifiers to be enabled.

In [ ]:
benchmark_table = pd.DataFrame(rows).sort_values(["Feature set", "f1_macro"], ascending=[True, False])
save_table(benchmark_table, ROOT, "benchmark_metrics.csv")
save_object(benchmark, ROOT, "benchmark.joblib")
display(benchmark_table.round(4))
print("Saved benchmark record, model files, and metrics. Test sources:", len(test_idx))